# Notebook 05 — Statistical Significance

**AAAI 2026 — Adaptive Methods for Class-Imbalanced Time-Series Classification**

> **Data policy:** If `experiments/raw_results.csv` exists, use it. Otherwise, generate synthetic mock results for demonstration.

In [ ]:
# ── Cell 1: Load results (mock fallback) ───────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_CSV = Path("../experiments/raw_results.csv")

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    print(f"Loaded {len(df)} runs from {RESULTS_CSV}")
else:
    # Synthetic mock results for demonstration
    rng = np.random.default_rng(42)
    methods = ['ce','weighted_ce','focal','class_balanced','ldam','logit_adj',
               'balanced_softmax','icmlt','adacal']
    datasets = ['stock','ucr_forda','ucr_electricdevices','ecg_mitbih']
    archs = ['lstm','inception_time','patchtst']
    seeds = [0, 1, 2]
    rows = []
    method_boost = {
        'ce': 0.0, 'weighted_ce': 0.03, 'focal': 0.05, 'class_balanced': 0.06,
        'ldam': 0.07, 'logit_adj': 0.06, 'balanced_softmax': 0.05,
        'icmlt': 0.07, 'adacal': 0.12
    }
    base_f1 = {'stock': 0.61, 'ucr_forda': 0.78, 'ucr_electricdevices': 0.55, 'ecg_mitbih': 0.72}
    for m in methods:
        for d in datasets:
            for a in archs:
                for s in seeds:
                    f1 = base_f1[d] + method_boost[m] + rng.normal(0, 0.015)
                    f1 = float(np.clip(f1, 0, 1))
                    ba = f1 - rng.uniform(0.01, 0.03)
                    gm = f1 - rng.uniform(0.01, 0.04)
                    rows.append({
                        'method': m, 'dataset': d, 'architecture': a, 'seed': s,
                        'macro_f1': round(f1, 4),
                        'balanced_accuracy': round(ba, 4),
                        'g_mean': round(gm, 4)
                    })
    df = pd.DataFrame(rows)
    print(f"Using synthetic mock data ({len(df)} rows). Run experiments to get real results.")

print(df.head())

In [ ]:
# ── Cell 2: Pairwise Wilcoxon signed-rank tests: AdaCAL vs each baseline ──
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

baselines = [m for m in df['method'].unique() if m != 'adacal']
datasets_list = df['dataset'].unique()
archs_list = df['architecture'].unique()
seeds_list = sorted(df['seed'].unique())

sig_rows = []

for ds in datasets_list:
    for arch in archs_list:
        # AdaCAL scores across seeds
        adacal_scores = []
        for s in seeds_list:
            r = df[(df['method']=='adacal') & (df['dataset']==ds) &
                   (df['architecture']==arch) & (df['seed']==s)]
            if not r.empty:
                adacal_scores.append(r['macro_f1'].values[0])

        for baseline in baselines:
            baseline_scores = []
            for s in seeds_list:
                r = df[(df['method']==baseline) & (df['dataset']==ds) &
                       (df['architecture']==arch) & (df['seed']==s)]
                if not r.empty:
                    baseline_scores.append(r['macro_f1'].values[0])

            if len(adacal_scores) >= 2 and len(baseline_scores) >= 2:
                n = min(len(adacal_scores), len(baseline_scores))
                x = np.array(adacal_scores[:n])
                y = np.array(baseline_scores[:n])

                # Wilcoxon signed-rank test (alternative: adacal > baseline)
                try:
                    stat, pval = stats.wilcoxon(x, y, alternative='greater')
                except ValueError:
                    # All differences are zero — fall back to two-sided
                    stat, pval = 1.0, 1.0

                # Effect size r = Z / sqrt(n)
                # Approximate Z from p-value (two-sided reference)
                z_approx = stats.norm.ppf(1 - pval) if pval < 1.0 else 0.0
                effect_r = abs(z_approx) / np.sqrt(n)

                mean_diff = float(np.mean(x) - np.mean(y))

                sig_rows.append({
                    'method': baseline,
                    'dataset': ds,
                    'architecture': arch,
                    'p_value': round(pval, 4),
                    'significant': pval < 0.05,
                    'effect_size_r': round(effect_r, 4),
                    'mean_f1_diff': round(mean_diff, 4)
                })

sig_df = pd.DataFrame(sig_rows)
print(f"Total comparisons: {len(sig_df)}")
print(f"Significant (p<0.05): {sig_df['significant'].sum()}")
display(sig_df.head(20))

In [ ]:
# ── Cell 3: Summary — how many combos show significant improvement? ────────
n_combos = len(datasets_list) * len(archs_list)  # total (dataset, arch) combos

summary = (
    sig_df.groupby('method')
    .agg(
        n_significant=('significant', 'sum'),
        n_total=('significant', 'count'),
        pct_significant=('significant', lambda x: 100 * x.mean()),
        mean_effect_r=('effect_size_r', 'mean'),
        mean_f1_diff=('mean_f1_diff', 'mean')
    )
    .reset_index()
    .sort_values('n_significant', ascending=False)
)
summary['pct_significant'] = summary['pct_significant'].round(1)
summary['mean_effect_r'] = summary['mean_effect_r'].round(3)
summary['mean_f1_diff'] = summary['mean_f1_diff'].round(4)

print(f"Summary: AdaCAL vs each baseline across {n_combos} (dataset × architecture) combos")
display(summary)

In [ ]:
# ── Cell 4: Heatmap — p-values per baseline × dataset, one subplot/arch ───
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

archs_sorted = sorted(archs_list)
baselines_sorted = sorted(baselines)
datasets_sorted = sorted(datasets_list)

fig, axes = plt.subplots(1, len(archs_sorted), figsize=(5 * len(archs_sorted), 4), sharey=True)
if len(archs_sorted) == 1:
    axes = [axes]

# Use a diverging colormap: green (p<0.05) → red (p>0.05)
cmap = mcolors.LinearSegmentedColormap.from_list(
    'sig', [(0, '#2ca02c'), (0.05, '#98df8a'), (0.051, '#ff9896'), (1.0, '#d62728')]
)

for ax, arch in zip(axes, archs_sorted):
    sub = sig_df[sig_df['architecture'] == arch]
    mat = np.ones((len(baselines_sorted), len(datasets_sorted)))
    for i, bl in enumerate(baselines_sorted):
        for j, ds in enumerate(datasets_sorted):
            r = sub[(sub['method'] == bl) & (sub['dataset'] == ds)]
            if not r.empty:
                mat[i, j] = r['p_value'].values[0]

    im = ax.imshow(mat, cmap=cmap, vmin=0.0, vmax=0.5, aspect='auto')
    ax.set_xticks(range(len(datasets_sorted)))
    ax.set_xticklabels(datasets_sorted, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(len(baselines_sorted)))
    ax.set_yticklabels(baselines_sorted, fontsize=9)
    ax.set_title(f'Arch: {arch}', fontsize=11)

    # Annotate cells
    for i in range(len(baselines_sorted)):
        for j in range(len(datasets_sorted)):
            pv = mat[i, j]
            marker = '*' if pv < 0.05 else ''
            ax.text(j, i, f'{pv:.2f}{marker}', ha='center', va='center', fontsize=7)

fig.colorbar(im, ax=axes[-1], label='p-value (green=significant p<0.05)')
fig.suptitle('Wilcoxon p-values: AdaCAL vs Baselines (green = significant improvement)', fontsize=12, y=1.02)
plt.tight_layout()

fig_path = Path('../paper/figures/significance_heatmap.png')
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Saved heatmap to {fig_path}")
plt.show()

In [ ]:
# ── Cell 5: Export significance table as LaTeX ─────────────────────────────
from pathlib import Path

method_names = {
    'ce': 'Cross-Entropy', 'weighted_ce': 'Weighted CE', 'focal': 'Focal Loss',
    'class_balanced': 'Class-Balanced', 'ldam': 'LDAM', 'logit_adj': 'Logit Adj.',
    'balanced_softmax': 'Balanced Softmax', 'icmlt': 'ICMLT',
}

# Build pivot: method rows, dataset columns, value = p_value (best arch or avg)
pv_avg = (
    sig_df.groupby(['method', 'dataset'])
    .agg(p_mean=('p_value', 'mean'), sig_count=('significant', 'sum'), total=('significant', 'count'))
    .reset_index()
)
pv_avg['sig_ratio'] = pv_avg.apply(lambda r: f"{int(r['sig_count'])}/{int(r['total'])}", axis=1)

lines = []
lines.append(r'\begin{table}[t]')
lines.append(r'\centering')
lines.append(r'\caption{Statistical significance: Wilcoxon signed-rank test (AdaCAL vs baseline).')
lines.append(r'Entries show mean $p$-value (sig/total architectures with $p < 0.05$).')
lines.append(r'$^*p<0.05$, $^{**}p<0.01$.}')
lines.append(r'\label{tab:significance}')
ds_cols = ' '.join(['c'] * len(datasets_sorted))
lines.append(r'\begin{tabular}{l' + ds_cols + r'}')
lines.append(r'\toprule')
header_cols = ' & '.join(datasets_sorted)
lines.append(f'Method & {header_cols} \\\\')
lines.append(r'\midrule')

for bl in baselines_sorted:
    disp = method_names.get(bl, bl)
    cells = [disp]
    for ds in datasets_sorted:
        r = pv_avg[(pv_avg['method'] == bl) & (pv_avg['dataset'] == ds)]
        if r.empty:
            cells.append('—')
        else:
            pv = r['p_mean'].values[0]
            sig_r = r['sig_ratio'].values[0]
            star = ''
            if pv < 0.01:
                star = '$^{**}$'
            elif pv < 0.05:
                star = '$^*$'
            cells.append(f'{pv:.3f}{star} ({sig_r})')
    lines.append(' & '.join(cells) + r' \\')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

latex_str = '\n'.join(lines)
out_path = Path('../paper/tables/significance.tex')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(latex_str)
print(f"LaTeX saved to {out_path}")
print()
print(latex_str)